# Section 5 — Frequent Itemsets (Apriori + FP-Growth) + Association Rules
**Course:** Data Mining
**Project:** FitZone Sports
**Reference:** `Section+5_Data+Mining_Frequent-Itemset.pdf`,
`Section_5_Frequent_Itemsets_using_Apriori_FB_Growth_Algorithms_3.html`

> Eng. Taher's task (التاسك رقم ٣): *Select a sample transaction dataset and
> apply the Apriori and FP-Growth algorithms to identify frequent itemsets.
> Subsequently, derive the Association Rules from the generated frequent
> itemsets.*

We use a **FitZone Sports market-basket dataset** (built directly in this
notebook so it runs anywhere). Each row is one customer's basket on a single
visit to the shop. To swap in a Kaggle dataset, replace the `transactions`
list with rows loaded from a CSV.

## 1. Setup

In [1]:
# !pip install mlxtend     # uncomment if mlxtend is not installed yet
import pandas as pd
from mlxtend.preprocessing     import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

## 2. Transactions dataset
Each row = one customer's basket at FitZone Sports. The dataset is hand-crafted
to expose realistic cross-sell patterns (e.g. football ball + football cleats,
running shoes + GPS watch, swim goggles + swimsuit) so the algorithms have
something interesting to find.

In [2]:
transactions = [
    # ─ Football customers ───────────────────────────────────────────
    ["Football Ball", "Football Cleats", "Sports Socks"],
    ["Football Ball", "Football Cleats", "Water Bottle"],
    ["Football Ball", "Football Cleats", "Sports Socks", "Water Bottle"],
    ["Football Ball", "Sports Socks"],
    ["Football Cleats", "Sports Socks", "Water Bottle"],

    # ─ Running customers ────────────────────────────────────────────
    ["Running Shoes", "GPS Watch", "Water Bottle"],
    ["Running Shoes", "GPS Watch", "Sports Socks"],
    ["Running Shoes", "GPS Watch", "Water Bottle", "Sports Socks"],
    ["Running Shoes", "Sports Socks"],
    ["Running Shoes", "GPS Watch"],

    # ─ Gym customers ────────────────────────────────────────────────
    ["Yoga Mat", "Dumbbell Set", "Water Bottle"],
    ["Yoga Mat", "Dumbbell Set"],
    ["Treadmill", "Dumbbell Set", "Water Bottle"],
    ["Yoga Mat", "Water Bottle"],
    ["Dumbbell Set", "Water Bottle", "Sports Socks"],

    # ─ Swimming customers ───────────────────────────────────────────
    ["Swim Goggles", "Swimsuit", "Swim Cap"],
    ["Swim Goggles", "Swimsuit"],
    ["Swim Goggles", "Swim Cap", "Water Bottle"],
    ["Swimsuit", "Swim Cap"],
    ["Swim Goggles", "Swimsuit", "Swim Cap", "Water Bottle"],

    # ─ Cycling customers ────────────────────────────────────────────
    ["Mountain Bike", "Cycling Helmet", "Water Bottle"],
    ["Mountain Bike", "Cycling Helmet"],
    ["Mountain Bike", "Cycling Helmet", "GPS Watch"],
    ["Cycling Helmet", "Sports Socks"],
    ["Mountain Bike", "Water Bottle", "Cycling Helmet"],
]
print(f"{len(transactions)} transactions, "
      f"{len({i for t in transactions for i in t})} distinct items")

25 transactions, 14 distinct items


## 3. One-hot encode (`TransactionEncoder`)

In [3]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)
df.head(10)

,Cycling Helmet,Dumbbell Set,Football Ball,Football Cleats,GPS Watch,Mountain Bike,Running Shoes,Sports Socks,Swim Cap,Swim Goggles,Swimsuit,Treadmill,Water Bottle,Yoga Mat
0,False,False,True,True,False,False,False,True,False,False,False,False,False,False
1,False,False,True,True,False,False,False,False,False,False,False,False,True,False
2,False,False,True,True,False,False,False,True,False,False,False,False,True,False
3,False,False,True,False,False,False,False,True,False,False,False,False,False,False
4,False,False,False,True,False,False,False,True,False,False,False,False,True,False
5,False,False,False,False,True,False,True,False,False,False,False,False,True,False
6,False,False,False,False,True,False,True,True,False,False,False,False,False,False
7,False,False,False,False,True,False,True,True,False,False,False,False,True,False
8,False,False,False,False,False,False,True,True,False,False,False,False,False,False
9,False,False,False,False,True,False,True,False,False,False,False,False,False,False


## 4. Apriori
- `min_support = 0.15` → an itemset must appear in ≥ 15% of baskets.
- `use_colnames=True` returns readable item names instead of column indices.

In [4]:
freq_ap = apriori(df, min_support=0.15, use_colnames=True)
freq_ap = freq_ap.sort_values('support', ascending=False).reset_index(drop=True)
print("Apriori frequent itemsets:", len(freq_ap))
freq_ap

Apriori frequent itemsets: 15


,support,itemsets
0,0.52,frozenset({Water Bottle})
1,0.36,frozenset({Sports Socks})
2,0.20,frozenset({Cycling Helmet})
3,0.20,frozenset({Running Shoes})
4,0.20,frozenset({GPS Watch})
5,0.16,frozenset({Football Ball})
6,0.16,frozenset({Dumbbell Set})
7,0.16,frozenset({Mountain Bike})
8,0.16,frozenset({Football Cleats})
9,0.16,frozenset({Swim Goggles})


### Filtering tricks shown in the lecture

In [5]:
freq_ap['length'] = freq_ap['itemsets'].apply(lambda s: len(s))

# Itemsets of length 2 with support >= 0.15
freq_ap[(freq_ap['length'] == 2) & (freq_ap['support'] >= 0.15)]

,support,itemsets,length
12,0.16,"frozenset({Mountain Bike, Cycling Helmet})",2
13,0.16,"frozenset({GPS Watch, Running Shoes})",2
14,0.16,"frozenset({Water Bottle, Sports Socks})",2


In [6]:
# Look for a specific itemset
freq_ap[freq_ap['itemsets'] == frozenset({'Football Ball', 'Football Cleats'})]

,support,itemsets,length


## 5. FP-Growth — same API, different internal algorithm

In [7]:
freq_fp = fpgrowth(df, min_support=0.15, use_colnames=True)
freq_fp = freq_fp.sort_values('support', ascending=False).reset_index(drop=True)
print("FP-Growth frequent itemsets:", len(freq_fp))
freq_fp

FP-Growth frequent itemsets: 15


,support,itemsets
0,0.52,frozenset({Water Bottle})
1,0.36,frozenset({Sports Socks})
2,0.20,frozenset({Running Shoes})
3,0.20,frozenset({GPS Watch})
4,0.20,frozenset({Cycling Helmet})
5,0.16,frozenset({Football Ball})
6,0.16,frozenset({Football Cleats})
7,0.16,frozenset({Dumbbell Set})
8,0.16,frozenset({Swim Cap})
9,0.16,frozenset({Swimsuit})


In [8]:
# Verify both algorithms returned the same set of frequent itemsets
ap_set = {frozenset(s) for s in freq_ap['itemsets']}
fp_set = {frozenset(s) for s in freq_fp['itemsets']}
print("Same itemsets ?", ap_set == fp_set)
print("Only in Apriori   :", ap_set - fp_set)
print("Only in FP-Growth :", fp_set - ap_set)

Same itemsets ? True
Only in Apriori   : set()
Only in FP-Growth : set()


## 6. Association rules
- **support**(A → C) = P(A ∩ C)
- **confidence**(A → C) = P(C | A) = support(A ∪ C) / support(A)
- **lift**(A → C) = confidence(A → C) / support(C). lift > 1 → positive
  correlation.

In [9]:
rules = association_rules(freq_ap, metric='confidence', min_threshold=0.6)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Mountain Bike}),frozenset({Cycling Helmet}),0.16,0.20,0.16,1.0,5.0,1.0,0.128,inf,0.952381,0.800000,1.000000,0.9
1,frozenset({Cycling Helmet}),frozenset({Mountain Bike}),0.20,0.16,0.16,0.8,5.0,1.0,0.128,4.2,1.000000,0.800000,0.761905,0.9
2,frozenset({GPS Watch}),frozenset({Running Shoes}),0.20,0.20,0.16,0.8,4.0,1.0,0.120,4.0,0.937500,0.666667,0.750000,0.8
3,frozenset({Running Shoes}),frozenset({GPS Watch}),0.20,0.20,0.16,0.8,4.0,1.0,0.120,4.0,0.937500,0.666667,0.750000,0.8


### Filter rules by user-defined criteria

In [10]:
rules['antecedent_len'] = rules['antecedents'].apply(len)
filtered = rules[(rules['antecedent_len'] >= 1) &
                 (rules['confidence']     > 0.7)  &
                 (rules['lift']           > 1.0)]
filtered.head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
0,frozenset({Mountain Bike}),frozenset({Cycling Helmet}),0.16,0.20,0.16,1.0,5.0,1.0,0.128,inf,0.952381,0.800000,1.000000,0.9,1
1,frozenset({Cycling Helmet}),frozenset({Mountain Bike}),0.20,0.16,0.16,0.8,5.0,1.0,0.128,4.2,1.000000,0.800000,0.761905,0.9,1
2,frozenset({GPS Watch}),frozenset({Running Shoes}),0.20,0.20,0.16,0.8,4.0,1.0,0.120,4.0,0.937500,0.666667,0.750000,0.8,1
3,frozenset({Running Shoes}),frozenset({GPS Watch}),0.20,0.20,0.16,0.8,4.0,1.0,0.120,4.0,0.937500,0.666667,0.750000,0.8,1


## 7. Interpretation

A few high-lift rules and what they mean for FitZone Sports (values vary
slightly with the dataset):

- `{Football Ball} → {Football Cleats}` — high confidence, lift > 1: a customer
  buying a football ball almost always also picks up a pair of cleats. Place
  these on adjacent shelves and bundle them in a cross-sell promo.
- `{Running Shoes} → {GPS Watch}` — runners pair their shoes with a GPS watch.
  Build a "Runner's Combo" landing page on the website.
- `{Mountain Bike} → {Cycling Helmet}` — every cycling customer should leave
  the store with a helmet; this rule confirms that bundling is already
  happening organically.
- `{Swim Goggles} → {Swimsuit}` — classic swimming-pair: train staff to
  ask "do you need a swimsuit too?".

Operational uses of the rules:

1. Place complementary products next to each other in-store.
2. Build cross-sell promotions (`buy A, get C 10% off`).
3. Re-design landing pages around inferred "personas"
   (Footballer / Runner / Cyclist / Swimmer / Gym-Goer).

---
**End of Section 5 notebook.** Apriori + FP-Growth produced identical frequent
itemsets, and the strongest rules were derived using both `confidence` and
`lift` thresholds.